In [ ]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
from pathlib import Path
import torch
import cv2
import os
import shutil
import sys

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

In [ ]:
detect_model = YOLO("models/11Lv2.pt")
pose_model = YOLO("yolo11m-pose")

In [ ]:
video_input_directory = Path("video/input")
video_output_directory = Path("video/output")

video_input_directory.mkdir(parents=True, exist_ok=True)
video_output_directory.mkdir(parents=True, exist_ok=True)

for item in video_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [ ]:
from ultralytics.utils.metrics import box_iou

for video_index, video in enumerate(video_input_directory.glob("clip4.mp4")):
    print(f"Video: {video}")

    detect_results = detect_model.track(
        device=0,
        source=video,
        tracker="botsort.yaml",
        persist=True,
        conf=0.5,
        stream=True,
        verbose=False,
    )

    cap = cv2.VideoCapture(str(video))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1
    cap.release()
    
    output_video = str(video_output_directory / f"video{str(video_index)}.mp4")
    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    output_log = str(video_output_directory / f"video_log{str(video_index)}.txt")
    with open(output_log, "w") as log_file:

        try:

            for result_index, result in enumerate(detect_results):
                
                something_happened = False
                est_time_seconds = int(result_index/fps)
                minutes, seconds = divmod(est_time_seconds, 60)
                formatted_time = f"{minutes:02}:{seconds:02}"
                log_line = f"Frame {result_index:05} @ {formatted_time} >> "

                print(f"\rProcessing Frame: {result_index}/{total_frames}", end="", flush=True)

                frame = result.orig_img.copy()
                boxes = result.boxes

                human_detected = False

                for box in boxes:
                    cls = detect_model.names[int(box.cls[0])]
                    if cls == "person":
                        human_detected = True
                
                if human_detected:
                    pose_results = pose_model(
                        source=frame,
                        show_labels=False,
                        device=0,
                        conf=0.5,
                        verbose=False
                    )
                    frame = pose_results[0].plot(boxes=False)

                for box in boxes:

                    xyxy = box.xyxy[0].cpu().numpy()
                    conf = round(float(box.conf[0]), 2)
                    cls = detect_model.names[int(box.cls[0])]
                    track_id = int(box.id[0]) if box.id is not None else None

                    x1, y1, x2, y2 = xyxy
                    rect_rgb = (255,0,0)
                    rect_thickness = 2
                    cv2.rectangle(
                        frame,
                        (int(x1), int(y1)),
                        (int(x2), int(y2)),
                        rect_rgb,
                        rect_thickness
                    )
                                    
                    font_scale = 1.0
                    font_rgb = (255,0,0)
                    font_thickness = 2
                    label = f"ID:{track_id} CLS:{cls} CONF:{conf}"
                    cv2.putText(
                        frame,
                        label,
                        (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        font_scale,
                        font_rgb,
                        font_thickness
                    )

                    if cls == "person":
                        for box2_index, box2 in enumerate(boxes):
                            if (box.xyxy == box2.xyxy).all():
                                continue

                            x1_p, y1_p, x2_p, y2_p = box.xyxy[0].cpu().numpy()
                            x1_o, y1_o, x2_o, y2_o = box2.xyxy[0].cpu().numpy()
                            xi1 = max(x1_p, x1_o)
                            yi1 = max(y1_p, y1_o)
                            xi2 = min(x2_p, x2_o)
                            yi2 = min(y2_p, y2_o)
                            inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
                            obj_area = max(0, x2_o - x1_o) * max(0, y2_o - y1_o)
                            ioa = inter_area / obj_area if obj_area > 0 else 0.0

                            if ioa < 0.5:
                                continue

                            cls2 = detect_model.names[int(box2.cls[0])]
                            if cls2 == "person":
                                continue
                            
                            track_id2 = int(box2.id[0]) if box2.id is not None else None
                            if something_happened:
                                log_line += " | "
                            log_line += f"{ioa:<7.4f} ID:{track_id:>04} {cls:<8} ID:{track_id2:>04} {cls2:<8}"
                            something_happened = True
                
                writer.write(frame)
                if something_happened:
                    log_file.write(log_line + "\n")

        except Exception as e:
            if writer is not None:
                writer.release()
            print(f"\n\nCrash detected!")
            print(e)

            sys.exit(1)
        
        print(f"\nCompleted {video}\n")
        writer.release()

print("All Done")